# Arm G layer-16 resolution

The seed-106 cross-layer run moved the causal mass from layer 27 to layer 16, and
the mathematical audit closed the selection and absolute-perturbation-size
challenges. Four questions remain before the layer-16 object is worth building a
multi-turn experiment on. This run answers all four on fresh seed-107 prompts.

**Primary — selectivity.** The equal-norm random control rules out generic damage
from removing four dimensions, but not the possibility that *any* task-structured
direction at layer 16 has leverage on the DECLINE−READ margin. Two structured null
subspaces are built by the identical paired construction and ablated identically:
the visible catalog control tag (KITE vs MOSS, which phase 1 showed the model
represents essentially perfectly and which is irrelevant to scope), and family
identity. A pre-specified disambiguator also ablates the component of each null
that is provably disjoint from the conflict subspace.

**Secondary.** Depth sweep 13–19 to test whether 16 is a peak or a shoulder; rank 1
vs rank 4 at layer 16, since ~78% of the seed-106 displacement was rank-1; and
final-position residual-stream norms, so relative rather than merely absolute
perturbation size can be evaluated.

Set **Runtime → Change runtime type → A100 GPU**, then run the cells in order.
Roughly 420 batched forward passes over 128 evaluation prompts; expect about
10–20 minutes on an A100. No generation, no checkpointing.

In [ ]:
# Colab supplies torch/CUDA.
print("Protocol: ARM_G_LAYER16_RESOLUTION_V1")
%pip -q install "transformers==5.0.0" "accelerate==1.12.0" \
  "sentence-transformers==5.2.2" "scikit-learn==1.8.0"

In [ ]:
# Mount Drive and copy the frozen launch files to local Colab storage.
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
LAUNCH_DIR = "/content/drive/MyDrive/phi-map/arm-g-layer16-launch"
LAUNCH_FILES = (
    "arm_g_layer16.py",
    "arm_g_cross_layer.py",
    "arm_g_causal_subspace.py",
    "arm_g_causal_dose_ablation.py",
    "arm_g_causal.py",
    "arm_g_phase1.py",
    "arm_g_scenarios.py",
)
for name in LAUNCH_FILES:
    source = f"{LAUNCH_DIR}/{name}"
    assert os.path.exists(source), f"Missing {source}"
    shutil.copy2(source, f"/content/{name}")
print("Arm G layer-16 launch files staged: OK")

In [ ]:
# Frozen run configuration.
ACTING_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
WORK_DIR = "/content/drive/MyDrive/phi-map/arm-g-layer16-seed107-v1"
PAIRS_PER_FAMILY = 16
BOOTSTRAP = 2000
RANDOM_SUBSPACES = 8
DEPTH_RANDOM_SUBSPACES = 4
BATCH_SIZE = 16

# Put HF_TOKEN in Colab's Secrets panel; do not paste it into the notebook.
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add an HF_TOKEN secret with Llama-3.1-8B-Instruct access"

In [ ]:
# Hardware and gated-model access gate.
from huggingface_hub import hf_hub_download
hf_hub_download(ACTING_MODEL, "config.json", token=HF_TOKEN)
print("Hugging Face model access: OK")

import subprocess, torch
subprocess.run(["nvidia-smi"], check=True)
props = torch.cuda.get_device_properties(0)
print(props.name, round(props.total_memory / 1024**3, 1), "GiB")
assert "A100" in props.name and props.total_memory >= 35 * 1024**3
assert torch.cuda.is_bf16_supported()

In [ ]:
# Deterministic protocol tests: basis construction for all three contrasts,
# hook ordering, and the frozen selectivity decision rule.
import os, subprocess, sys
env = dict(os.environ)
env["HF_TOKEN"] = HF_TOKEN
base_cmd = [
    sys.executable, "/content/arm_g_layer16.py",
    "--model", ACTING_MODEL,
    "--output-dir", WORK_DIR,
    "--pairs-per-family", str(PAIRS_PER_FAMILY),
    "--bootstrap", str(BOOTSTRAP),
    "--random-subspaces", str(RANDOM_SUBSPACES),
    "--depth-random-subspaces", str(DEPTH_RANDOM_SUBSPACES),
    "--batch-size", str(BATCH_SIZE),
]
subprocess.run(base_cmd + ["--self-test"], check=True, env=env)

In [ ]:
# Depth sweep, rank sweep, structured null contrasts, and random controls.
subprocess.run(base_cmd, check=True, env=env)

In [ ]:
# Compact result view. Full per-row data remains in Drive.
import json
result_path = f"{WORK_DIR}/arm_g_layer16_result.json"
result = json.load(open(result_path))
sel = result["selectivity"]
summary = {
    "decision": result["decision"],
    "decision_reasons": result["decision_reasons"],
    "baseline_condition_contrast": result["baseline"]["condition_contrast"]["overall"],
    "selectivity": {
        "conflict_attenuation": sel["conflict_attenuation"],
        "null_raw": {
            k: v["attenuation"]["overall"] for k, v in sel["null_contrasts"].items()
        },
        "null_orthogonalized": {
            k: v["attenuation"]["overall"]
            for k, v in sel["null_contrasts_orthogonalized"].items()
        },
        "conflict_minus_null": sel["conflict_minus_null"],
        "conflict_minus_null_orthogonalized": sel["conflict_minus_null_orthogonalized"],
        "principal_angles": {
            k: v["principal_angle_cosines_with_conflict"]
            for k, v in sel["null_contrasts"].items()
        },
        "random_absolute_p95": sel["random_absolute_p95"],
    },
    "depth": {
        layer: {
            "attenuation": item["attenuation"]["overall"],
            "ci_95": item["attenuation_bootstrap"]["ci_95"],
            "displacement_over_residual_norm": item["displacement_over_residual_norm"],
            "random_p95": result["depth_sweep"]["random_controls"][layer]["absolute_p95"],
        }
        for layer, item in result["depth_sweep"]["layers"].items()
    },
    "peak_layer": result["depth_sweep"]["peak_layer"],
    "focus_layer_is_peak": result["depth_sweep"]["focus_layer_is_peak"],
    "ranks": {
        r: item["attenuation"]["overall"]
        for r, item in result["rank_sweep"]["ranks"].items()
    },
    "rank4_minus_rank1": result["rank_sweep"]["rank4_minus_rank1"],
    "residual_norms": result["residual_norms"],
    "seed_replication": result["seed_replication"],
}
print(json.dumps(summary, indent=2))

In [ ]:
# Archive a compact summary and a compressed full artifact for the repo.
import base64, gzip, json

summary_path = f"{WORK_DIR}/arm_g_layer16_result_summary.json"
archive_path = f"{WORK_DIR}/arm_g_layer16_result.json.gz.b64"
with open(summary_path, "w") as handle:
    json.dump(
        {
            **summary,
            "protocol": {
                "source_seeds": [101, 102],
                "evaluation_seed": 107,
                "depth_layers": [13, 14, 15, 16, 17, 18, 19],
                "focus_layer": 16,
                "primary_rank": 4,
                "null_contrasts": ["control_tag", "family"],
                "bootstrap_repetitions": BOOTSTRAP,
            },
            "sample_counts": result["sample_counts"],
            "full_result_artifact": "arm_g_layer16_result.json.gz.b64",
        },
        handle,
        indent=1,
    )
raw = json.dumps(result).encode("utf-8")
with open(archive_path, "wb") as handle:
    handle.write(base64.b64encode(gzip.compress(raw)))
print("summary:", summary_path)
print("archive:", archive_path)